In [1]:
import pandas as pd
import numpy as np
import time

print("Inizializzazione Motore di Backtest - Setup 1: Assorbimento...")
start_time = time.time()

# --- 1. CARICAMENTO DATI ---
# Percorso corretto per il tuo Mac M4
percorso_parquet = "/Users/leonardosposato/Documents/git/QuantEdge_Project/data/processed/es_master_4months.parquet"

print(f"Caricamento dataset da: {percorso_parquet}...")
# Carichiamo i dati eliminando immediatamente i NaN finali
df = pd.read_parquet(percorso_parquet).dropna(subset=['bid', 'ask'])

# Impostiamo il tempo come indice e lo ordiniamo (fondamentale per le operazioni Rolling)
df = df.set_index('Datetime_UTC')
df = df.sort_index()
print(f"Dati caricati. Righe operative: {len(df):,}")

# --- 2. PARAMETRI DELLA STRATEGIA (Da Calibrare) ---
# Questi sono i parametri suggeriti da Claude, che in seguito ottimizzeremo
FINESTRA_MS = '5S'           # Finestra di osservazione (5 secondi)
SOGLIA_VOLUME = 500          # Volume totale minimo scambiato nei 5 secondi
SOGLIA_DELTA = 300           # Delta minimo (sbilanciamento) nei 5 secondi
MAX_TICK_MOUVEMENT = 1.00    # Il prezzo non deve muoversi più di 1 punto intero (4 tick)

# --- 3. COSTRUZIONE DEI SEGNALI (VETTORIALE) ---
print("Calcolo aggregazioni volumetriche (Rolling Window)...")

# Calcoliamo il volume cumulativo e il delta cumulativo ogni 5 secondi
df['Rolling_Volume'] = df['Volume'].rolling(window=FINESTRA_MS).sum()
df['Rolling_Delta'] = df['Delta'].rolling(window=FINESTRA_MS).sum()

# Calcoliamo il movimento massimo del prezzo negli ultimi 5 secondi
df['Rolling_Max_Price'] = df['Price'].rolling(window=FINESTRA_MS).max()
df['Rolling_Min_Price'] = df['Price'].rolling(window=FINESTRA_MS).min()
df['Price_Range'] = df['Rolling_Max_Price'] - df['Rolling_Min_Price']

print("Ricerca dei Trigger di Assorbimento...")

# TRIGGER LONG: Troviamo quando i Venditori Aggressivi vengono Assorbiti
# 1. Tanti volumi venduti (Delta negativo pesante)
# 2. Volumi totali alti
# 3. Il prezzo non è crollato (Price Range piccolo)
df['Signal_Long'] = np.where(
    (df['Rolling_Delta'] <= -SOGLIA_DELTA) & 
    (df['Rolling_Volume'] >= SOGLIA_VOLUME) & 
    (df['Price_Range'] <= MAX_TICK_MOUVEMENT),
    1, 0
)

# TRIGGER SHORT: Troviamo quando i Compratori Aggressivi vengono Assorbiti
# 1. Tanti volumi comprati (Delta positivo pesante)
# 2. Volumi totali alti
# 3. Il prezzo non è esploso (Price Range piccolo)
df['Signal_Short'] = np.where(
    (df['Rolling_Delta'] >= SOGLIA_DELTA) & 
    (df['Rolling_Volume'] >= SOGLIA_VOLUME) & 
    (df['Price_Range'] <= MAX_TICK_MOUVEMENT),
    -1, 0
)

# Uniamo i segnali in una singola colonna (-1 per Short, 0 per Nessun Segnale, 1 per Long)
df['Signal'] = df['Signal_Long'] + df['Signal_Short']

# Isoliamo solo gli istanti esatti in cui scatta il segnale
trade_signals = df[df['Signal'] != 0].copy()

print(f"\n--- SCANSIONE COMPLETATA ---")
print(f"Trovati {len(trade_signals)} potenziali setup di Assorbimento Istituzionale.")

# Pulizia memoria (eliminiamo dal dataframe originale le colonne pesanti che non servono più)
df = df.drop(columns=['Rolling_Volume', 'Rolling_Delta', 'Rolling_Max_Price', 'Rolling_Min_Price', 'Price_Range', 'Signal_Long', 'Signal_Short'])

end_time = time.time()
print(f"Tempo totale di calcolo: {(end_time - start_time):.2f} secondi\n")

# Mostriamo le prime 10 anomalie trovate
display(trade_signals[['Price', 'Rolling_Volume', 'Rolling_Delta', 'Price_Range', 'Signal', 'bid', 'ask']].head(10))

Inizializzazione Motore di Backtest - Setup 1: Assorbimento...
Caricamento dataset da: /Users/leonardosposato/Documents/git/QuantEdge_Project/data/processed/es_master_4months.parquet...
Dati caricati. Righe operative: 71,651,488
Calcolo aggregazioni volumetriche (Rolling Window)...


/var/folders/_t/rpmwn01n49vb6gd6h0vtn4zw0000gn/T/ipykernel_78483/3904314216.py:32: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  df['Rolling_Volume'] = df['Volume'].rolling(window=FINESTRA_MS).sum()
/var/folders/_t/rpmwn01n49vb6gd6h0vtn4zw0000gn/T/ipykernel_78483/3904314216.py:33: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  df['Rolling_Delta'] = df['Delta'].rolling(window=FINESTRA_MS).sum()
/var/folders/_t/rpmwn01n49vb6gd6h0vtn4zw0000gn/T/ipykernel_78483/3904314216.py:36: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  df['Rolling_Max_Price'] = df['Price'].rolling(window=FINESTRA_MS).max()
/var/folders/_t/rpmwn01n49vb6gd6h0vtn4zw0000gn/T/ipykernel_78483/3904314216.py:37: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  df['Rolling_Min_Price'] = df['Price'].rolling(window=FINES

Ricerca dei Trigger di Assorbimento...

--- SCANSIONE COMPLETATA ---
Trovati 104924 potenziali setup di Assorbimento Istituzionale.
Tempo totale di calcolo: 17.15 secondi



,Price,Rolling_Volume,Rolling_Delta,Price_Range,Signal,bid,ask
Datetime_UTC,,,,,,,
2026-02-09 13:15:55+00:00,6952.75,502.0,-416.0,0.75,1,6917.67,6917.92
2026-02-09 13:15:55+00:00,6952.75,503.0,-415.0,0.75,1,6917.67,6917.92
2026-02-09 13:15:55+00:00,6952.75,504.0,-414.0,0.75,1,6917.67,6917.92
2026-02-09 13:15:55+00:00,6952.75,505.0,-413.0,0.75,1,6917.67,6917.92
2026-02-09 13:15:55+00:00,6952.75,507.0,-415.0,0.75,1,6917.67,6917.92
2026-02-09 13:15:55+00:00,6952.75,508.0,-416.0,0.75,1,6917.67,6917.92
2026-02-09 13:15:55+00:00,6952.75,509.0,-417.0,0.75,1,6917.67,6917.92
2026-02-09 13:15:55+00:00,6952.75,510.0,-418.0,0.75,1,6917.67,6917.92
2026-02-09 13:15:55+00:00,6952.75,512.0,-420.0,0.75,1,6917.67,6917.92


In [ ]:
import pandas as pd
import numpy as np
import time

print("Avvio Motore di Esecuzione e Risk Management...")
start_sim = time.time()

# --- 1. APPLICAZIONE DEL COOLDOWN (Filtro dei Cluster) ---
COOLDOWN_SEC = 10  # Secondi di pausa obbligatoria dopo un trade

# Resettiamo l'indice per calcolare la differenza di tempo
signals_df = trade_signals.reset_index()

valid_trades = []
last_trade_time = None  # FIX: Usiamo None al posto dell'anno 1677 per evitare l'Overflow

for idx, row in signals_df.iterrows():
    # Se è il primo trade in assoluto, o se è passato il tempo di Cooldown
    if last_trade_time is None or (row['Datetime_UTC'] - last_trade_time).total_seconds() >= COOLDOWN_SEC:
        valid_trades.append(row)
        last_trade_time = row['Datetime_UTC']

df_trades = pd.DataFrame(valid_trades)
print(f"Segnali Grezzi: {len(trade_signals):,} -> Trade Effettivi (dopo Cooldown): {len(df_trades):,}")

# --- 2. RISK ENGINE (SIMULAZIONE SL, TP E TIME STOP) ---
print("Simulazione esecuzione trades sul CFD (Applicazione Spread, TP, SL)...")

# Parametri operativi
TP_PUNTI = 1.50
SL_PUNTI = 1.00
TIME_STOP_SEC = 10

# Liste per salvare i risultati
esiti = []
pnl_list = []

for _, trade in df_trades.iterrows():
    t_ingresso = trade['Datetime_UTC']
    t_scadenza = t_ingresso + pd.Timedelta(seconds=TIME_STOP_SEC)
    direzione = trade['Signal']
    
    # Isola la fetta di mercato (i 10 secondi successivi al segnale)
    finestra_futura = df.loc[t_ingresso : t_scadenza]
    
    if finestra_futura.empty:
        continue
        
    # LOGICA LONG
    if direzione == 1:
        prezzo_ingresso = trade['ask'] # Compriamo sull'Ask (più alto)
        target_price = prezzo_ingresso + TP_PUNTI
        stop_price = prezzo_ingresso - SL_PUNTI
        
        hit_tp = (finestra_futura['bid'] >= target_price).idxmax() if (finestra_futura['bid'] >= target_price).any() else None
        hit_sl = (finestra_futura['bid'] <= stop_price).idxmax() if (finestra_futura['bid'] <= stop_price).any() else None
        
        if hit_tp and hit_sl:
            if hit_tp < hit_sl:
                esiti.append('WIN')
                pnl_list.append(TP_PUNTI)
            else:
                esiti.append('LOSS')
                pnl_list.append(-SL_PUNTI)
        elif hit_tp:
            esiti.append('WIN')
            pnl_list.append(TP_PUNTI)
        elif hit_sl:
            esiti.append('LOSS')
            pnl_list.append(-SL_PUNTI)
        else:
            prezzo_uscita = finestra_futura['bid'].iloc[-1]
            esiti.append('TIME_STOP')
            pnl_list.append(prezzo_uscita - prezzo_ingresso)
            
    # LOGICA SHORT
    elif direzione == -1:
        prezzo_ingresso = trade['bid'] # Vendiamo sul Bid (più basso)
        target_price = prezzo_ingresso - TP_PUNTI
        stop_price = prezzo_ingresso + SL_PUNTI
        
        hit_tp = (finestra_futura['ask'] <= target_price).idxmax() if (finestra_futura['ask'] <= target_price).any() else None
        hit_sl = (finestra_futura['ask'] >= stop_price).idxmax() if (finestra_futura['ask'] >= stop_price).any() else None
        
        if hit_tp and hit_sl:
            if hit_tp < hit_sl:
                esiti.append('WIN')
                pnl_list.append(TP_PUNTI)
            else:
                esiti.append('LOSS')
                pnl_list.append(-SL_PUNTI)
        elif hit_tp:
            esiti.append('WIN')
            pnl_list.append(TP_PUNTI)
        elif hit_sl:
            esiti.append('LOSS')
            pnl_list.append(-SL_PUNTI)
        else:
            prezzo_uscita = finestra_futura['ask'].iloc[-1]
            esiti.append('TIME_STOP')
            pnl_list.append(prezzo_ingresso - prezzo_uscita)

df_trades['Esito'] = esiti
df_trades['PnL_Punti'] = pnl_list

# --- 3. REPORT FINALE ---
win_rate = (len(df_trades[df_trades['Esito'] == 'WIN']) / len(df_trades)) * 100
tot_pnl = sum(pnl_list)
gross_profit = tot_pnl * 50 # 1 punto ES = 50 dollari

print(f"\n--- REPORT BACKTEST (4 MESI) ---")
print(f"Trade Eseguiti (dopo Cooldown): {len(df_trades)}")
print(f"Vittorie (TP Hit): {len(df_trades[df_trades['Esito'] == 'WIN'])}")
print(f"Sconfitte (SL Hit): {len(df_trades[df_trades['Esito'] == 'LOSS'])}")
print(f"Uscite per Tempo (Time Stop): {len(df_trades[df_trades['Esito'] == 'TIME_STOP'])}")
print(f"Win Rate Reale: {win_rate:.2f}%")
print(f"Net PnL (in punti indice): {tot_pnl:.2f} Punti")
print(f"Profitto Stimato (1 Lotto ES): ${gross_profit:,.2f}")
print(f"Tempo di simulazione: {(time.time() - start_sim):.2f} sec")

Avvio Motore di Esecuzione e Risk Management...


OutOfBoundsDatetime: Result is too large for pandas.Timedelta. Convert inputs to datetime.datetime with 'Timestamp.to_pydatetime()' before subtracting.